# AD Purple-Team Hunting Notebook

Exploratory hunting queries against the lab's Elasticsearch SIEM
(`telemetry/elastic/`), one section per `attack/techniques.py` technique.
These are **hunting** queries — broader and more exploratory than the
precise `detections/sigma/*.yml` rules — meant for a human to review
results and judge context, not to alert automatically.

Each section notes whether a Sigma rule already exists for this technique
(`detections/coverage_matrix.json`) — if it does, this notebook's job is
to catch variants/near-misses the rule's exact field-match wouldn't, not
to duplicate it.

**STATUS: written, not run** — no live Elasticsearch cluster exists yet
(no lab has been provisioned — see `ROADMAP.md`). Treat every query here
as unexecuted and unverified against real data; the DSL syntax is
consistent with `telemetry/dashboards/baseline-queries.md`'s hand-checked
queries, but nothing below has actually returned results from a real
index.

## Setup

In [ ]:
import json
import os

from elasticsearch import Elasticsearch

ES_HOST = os.environ.get("ES_HOST", "https://siem01.eadadl.lab:9200")
ES_USER = os.environ.get("ES_USER", "elastic")
ES_PASSWORD = os.environ["SIEM_ADMIN_PASSWORD"]  # see .env.example

es = Elasticsearch(ES_HOST, basic_auth=(ES_USER, ES_PASSWORD), verify_certs=True)
INDEX = "winlogbeat-*"

def hunt(query: dict, size: int = 50) -> list[dict]:
    """Run a query, return the hits' _source docs."""
    result = es.search(index=INDEX, query=query, size=size)
    return [hit["_source"] for hit in result["hits"]["hits"]]

LOOKBACK = {"range": {"@timestamp": {"gte": "now-7d"}}}

## Hunt 1: Kerberoasting variants

Sigma rule exists: `detections/sigma/kerberoasting.yml` (RC4 exact-match,
excludes machine accounts). This hunt is broader: **any** non-AES ticket
encryption type, not just RC4 (0x17) — catches DES (0x1/0x3, effectively
never legitimate in a modern domain and an even stronger signal than RC4),
and doesn't pre-exclude machine accounts, so a human can judge those cases
individually rather than have the rule silently drop them.

In [ ]:
AES_ETYPES = ["0x12", "0x11"]  # AES256, AES128 — everything else is weak

weak_ticket_query = {
    "bool": {
        "filter": [{"term": {"winlog.event_id": "4769"}}, LOOKBACK],
        "must_not": [{"terms": {"winlog.event_data.TicketEncryptionType": AES_ETYPES}}],
    }
}
results = hunt(weak_ticket_query)
print(f"{len(results)} non-AES service ticket requests in the last 7 days")
for r in results[:10]:
    event_data = r.get("winlog", {}).get("event_data", {})
    print(event_data.get("ServiceName"), event_data.get("TicketEncryptionType"))

## Hunt 2: AS-REP roasting variants

Sigma rule exists: `detections/sigma/asrep_roasting.yml`. This hunt
additionally surfaces **which accounts have ever** had a `PreAuthType: 0`
event, aggregated — a single unexpected account showing up here, even
outside an active alert window, is worth a standing review of its
`DoesNotRequirePreAuth` flag.

In [ ]:
asrep_agg_query = {
    "bool": {
        "filter": [
            {"term": {"winlog.event_id": "4768"}},
            {"term": {"winlog.event_data.PreAuthType": "0"}},
        ]
    }
}
result = es.search(
    index=INDEX,
    query=asrep_agg_query,
    size=0,
    aggs={"by_account": {"terms": {"field": "winlog.event_data.TargetUserName", "size": 50}}},
)
for bucket in result["aggregations"]["by_account"]["buckets"]:
    print(bucket["key"], bucket["doc_count"])

## Hunt 3: DCSync — replication rights from unexpected principals

Sigma rule exists: `detections/sigma/dcsync.yml` (excludes only `dc01$`).
This hunt lists **every distinct principal** that has ever exercised
`DS-Replication-Get-Changes[-All]`, not just recent events — a slow,
low-and-slow DCSync attempt (occasional single requests, not a burst)
could sit below a time-windowed alert threshold but still show up in a
full-history aggregation like this one.

In [ ]:
DCSYNC_GUIDS = [
    "1131f6aa-9c07-11d1-f79f-00c04fc2dcd2",
    "1131f6ad-9c07-11d1-f79f-00c04fc2dcd2",
]

dcsync_query = {
    "bool": {
        "filter": [
            {"term": {"winlog.event_id": "4662"}},
            {
                "bool": {
                    "should": [
                        {"wildcard": {"winlog.event_data.Properties": f"*{guid}*"}}
                        for guid in DCSYNC_GUIDS
                    ]
                }
            },
        ]
    }
}
result = es.search(
    index=INDEX,
    query=dcsync_query,
    size=0,
    aggs={
        "by_principal": {"terms": {"field": "winlog.event_data.SubjectUserName", "size": 50}}
    },
)
for bucket in result["aggregations"]["by_principal"]["buckets"]:
    flag = "" if bucket["key"] == "DC01$" else "  <-- REVIEW: not the known-legitimate DC"
    print(bucket["key"], bucket["doc_count"], flag)

## Hunt 4: Unconstrained delegation coercion (pipe access)

Sigma rule exists: `detections/sigma/unconstrained_delegation_coerce.yml`.
This hunt widens the pipe-name list beyond `efsrpc`/`lsarpc` to other
named pipes associated with published coercion techniques
(`netlogon` for Zerologon-adjacent activity, `spoolss` for
PrinterBug/SpoolSample) — broader net, more false positives expected, by
design for a hunt rather than an alerting rule.

In [ ]:
coercion_pipes = ["efsrpc", "lsarpc", "netlogon", "spoolss"]

pipe_should_clauses = [
    {"wildcard": {"winlog.event_data.PipeName": f"*{p}*"}} for p in coercion_pipes
]
pipe_query = {
    "bool": {
        "filter": [
            {"terms": {"winlog.event_id": ["17", "18"]}},
            {"bool": {"should": pipe_should_clauses}},
            LOOKBACK,
        ]
    }
}
results = hunt(pipe_query)
print(f"{len(results)} coercion-relevant pipe events in the last 7 days")

## Hunt 5: AD recon tooling (no dedicated Sigma rule beyond `bloodhound_collect`)

`detections/sigma/bloodhound_collect.yml` matches specific command-line
signatures. This hunt looks for the *effect* instead — a single host
generating an unusually high volume of LDAP-adjacent network connections
in a short window, which any collection tool (not just BloodHound/
SharpHound by name) would produce.

In [ ]:
ldap_volume_query = {
    "bool": {
        "filter": [
            {"term": {"winlog.event_id": "3"}},
            {"term": {"winlog.event_data.DestinationPort": "389"}},
            LOOKBACK,
        ]
    }
}
result = es.search(
    index=INDEX,
    query=ldap_volume_query,
    size=0,
    aggs={"by_source": {"terms": {"field": "winlog.event_data.SourceIp", "size": 20}}},
)
for bucket in result["aggregations"]["by_source"]["buckets"]:
    flag = "  <-- high volume, review" if bucket["doc_count"] > 100 else ""
    print(bucket["key"], bucket["doc_count"], flag)

## Cross-reference with detection coverage

Sanity check: which of the technique IDs hunted above already have a
passing Sigma rule (`make detections-test`), vs. which are hunt-only?

In [ ]:
with open("../../detections/coverage_matrix.json") as f:
    coverage = json.load(f)

for t in coverage["techniques"]:
    status = "-> covered by Sigma rule" if t["covered"] else "-> hunt-only, no passing rule"
    print(t["technique_id"], status)